# 03. Análisis bivariado

Este notebook ejecuta el análisis bivariado actualizado del proyecto. Las variables continuas se comparan entre participantes con salud ósea normal y alteración ósea mediante U de Mann-Whitney y delta de Cliff. Las variables categóricas se evalúan con Chi-cuadrada o prueba exacta de Fisher, y se reporta V de Cramer como tamaño de efecto.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / "src"))

from src.export_tables_pdf import (
    build_continuous_bivariate_table,
    prepare_grouped_categorical_table,
)

pd = __import__("pandas")
pd.set_option("display.max_columns", None)

In [2]:
PARQUET_PATH = PROJECT_DIR / "data" / "processed" / "BD_Clean_Osteoporosis.parquet"
PICKLE_PATH = PARQUET_PATH.with_suffix(".pkl")

if PARQUET_PATH.exists():
    df_clean = pd.read_parquet(PARQUET_PATH)
    loaded_path = PARQUET_PATH
elif PICKLE_PATH.exists():
    df_clean = pd.read_pickle(PICKLE_PATH)
    loaded_path = PICKLE_PATH
else:
    raise FileNotFoundError(
        "No se encontro la base limpia. Ejecuta primero el notebook 01_data_cleaning.ipynb. "
        f"Rutas revisadas: {PARQUET_PATH} y {PICKLE_PATH}"
    )

print(f"Dataset cargado desde: {loaded_path}")
print(f"Dimensiones de la base analitica: {df_clean.shape[0]:,} filas x {df_clean.shape[1]:,} columnas")

Dataset cargado desde: C:\Users\marco\Documents\analysis_osteoporosis\data\processed\BD_Clean_Osteoporosis.parquet
Dimensiones de la base analitica: 405 filas x 75 columnas


## Variables continuas

In [3]:
tabla_continuas = build_continuous_bivariate_table(df_clean)
tabla_continuas

,Variable,"Alteración ósea\nMediana (Q1, Q3)","Salud ósea normal\nMediana (Q1, Q3)",U,p-value,Delta de Cliff,Magnitud
0,Edad (años),"66.00 (60.75, 74.00)","57.00 (53.00, 64.00)",15857.5,< 0.001,0.511,Grande
1,Peso (kg),"68.00 (60.75, 74.00)","86.00 (76.00, 94.00)",2833.0,< 0.001,-0.730,Grande
2,Altura (cm),"154.50 (150.00, 160.00)","160.00 (154.00, 166.00)",6977.5,< 0.001,-0.335,Mediano
3,IMC,"27.56 (24.71, 30.41)","32.27 (30.36, 35.58)",4028.0,< 0.001,-0.616,Grande


## Variables categóricas

In [4]:
tabla_categoricas = prepare_grouped_categorical_table(df_clean)
tabla_categoricas.drop(columns=['_section'])

,Variable,Categoría,"Total (n, %)","Con alteración ósea (n, %)","Sin alteración ósea (n, %)",p-value,V de Cramer
0,Características demográficas,,,,,,
1,Edad,50-59,111 (27.4),77 (69.4),34 (30.6),< 0.001,0.298
2,,60 a 69,162 (40.0),139 (85.8),23 (14.2),,
3,,70 y más,132 (32.6),128 (97.0),4 (3.0),,
4,Sexo,Hombre,78 (19.3),64 (82.1),14 (17.9),0.5371,0.031
5,,Mujer,327 (80.7),280 (85.6),47 (14.4),,
6,Estado civil,En pareja,230 (56.8),187 (81.3),43 (18.7),,
7,,Sin pareja,175 (43.2),157 (89.7),18 (10.3),,
8,Lugar de residencia,Municipios de Jalisco,210 (51.9),178 (84.8),32 (15.2),1.0000,0.000
9,,Zona Metropolitana de Guadalajara,195 (48.1),166 (85.1),29 (14.9),,


Las salidas definitivas en CSV y PDF se regeneran desde `src/export_tables_pdf.py`, que utiliza estas mismas funciones.